In [1]:
import sys
import csv
import statistics
from collections import Counter

In [2]:
DEFAULT_CSV_PATH = "fb_ads_president_scored_anon.csv"

In [3]:
RANGE_COLUMNS = ["estimated_audience_size", "impressions", "spend"]

FLAG_COLUMNS = [
    "illuminating_scam", "illuminating_election_integrity_Truth",
    "illuminating_msg_type_advocacy", "illuminating_msg_type_issue",
    "illuminating_msg_type_attack", "illuminating_msg_type_image",
    "illuminating_msg_type_cta", "illuminating_cta_subtype_engagement",
    "illuminating_cta_subtype_fundraising", "illuminating_cta_subtype_voting",
    "illuminating_topic_covid", "illuminating_topic_economy",
    "illuminating_topic_education", "illuminating_topic_environment",
    "illuminating_topic_foreign_policy", "illuminating_topic_governance",
    "illuminating_topic_health", "illuminating_topic_immigration",
    "illuminating_topic_lgbtq_issues", "illuminating_topic_military",
    "illuminating_topic_race_and_ethnicity", "illuminating_topic_safety",
    "illuminating_topic_social_and_cultural",
    "illuminating_topic_technology_and_privacy",
    "illuminating_topic_womens_issue", "illuminating_incivility",
]

NUMERIC_COLUMNS = RANGE_COLUMNS + FLAG_COLUMNS

In [4]:
def parse_range(text):
    """Turn "{'lower_bound': '200', 'upper_bound': '299'}" into 249.5.
    Some rows only have one of the two keys, e.g. "{'lower_bound': '1000001'}"
    -- in that case we just use whichever number is present."""
    text = text.replace("'", "").replace("{", "").replace("}", "")
    lower = None
    upper = None
    for part in text.split(","):
        key, _, value = part.partition(":")
        key = key.strip()
        value = value.strip()
        if value == "":
            continue
        if key == "lower_bound":
            lower = float(value)
        elif key == "upper_bound":
            upper = float(value)
 
    if lower is None and upper is None:
        return None
    if lower is None:
        return upper
    if upper is None:
        return lower
    return (lower + upper) / 2

In [5]:
def to_number(column, text):
    """Convert one cell to a float, or return None if it's missing/bad."""
    if text is None or text.strip() == "":
        return None
    if column in RANGE_COLUMNS:
        return parse_range(text)
    try:
        return float(text)
    except ValueError:
        return None

In [6]:
def numeric_stats(values):
    """values is a list of floats (no Nones). Returns a dict of stats."""
    return {
        "count": len(values),
        "mean": sum(values) / len(values),
        "min": min(values),
        "max": max(values),
        "median": statistics.median(values),
        "stdev": statistics.stdev(values) if len(values) > 1 else 0,
    }

In [7]:
def categorical_stats(values):
    """values is a list of strings (no missing). Returns a dict of stats."""
    counts = Counter(values)
    return {
        "count": len(values),
        "unique": len(counts),
        "top5": counts.most_common(5),
    }

In [8]:
def main(path=None):
    if path is None:
        path = DEFAULT_CSV_PATH
 
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        rows = list(reader)
        columns = reader.fieldnames
 
    print(f"Rows: {len(rows):,}")
    print(f"Columns: {len(columns)}")
 
    for column in columns:
        print(f"\n--- {column} ---")
 
        if column in NUMERIC_COLUMNS:
            numbers = []
            missing = 0
            for row in rows:
                num = to_number(column, row[column])
                if num is None:
                    missing += 1
                else:
                    numbers.append(num)
 
            print(f"missing: {missing}")
            if numbers:
                stats = numeric_stats(numbers)
                print(f"count : {stats['count']:,}")
                print(f"mean  : {stats['mean']:.2f}")
                print(f"min   : {stats['min']:.2f}")
                print(f"max   : {stats['max']:.2f}")
                print(f"median: {stats['median']:.2f}")
                print(f"stdev : {stats['stdev']:.2f}")
 
        else:
            values = [row[column] for row in rows if row[column].strip() != ""]
            missing = len(rows) - len(values)
 
            print(f"missing: {missing}")
            if values:
                stats = categorical_stats(values)
                print(f"count : {stats['count']:,}")
                print(f"unique: {stats['unique']:,}")
                print("top 5 :")
                for value, freq in stats["top5"]:
                    short = value if len(value) <= 40 else value[:37] + "..."
                    print(f"    {short:40s} {freq:,}")

In [9]:
if __name__ == "__main__":
    if len(sys.argv) > 1 and not sys.argv[1].startswith("-"):
        main(sys.argv[1])
    else:
        main()

Rows: 246,745
Columns: 40

--- page_id ---
missing: 0
count : 246,745
unique: 4,475
top 5 :
    page_id_24413227922                      55,503
    page_id_153080620724                     23,988
    page_id_7860876103                       14,822
    page_id_255594894306221                  10,461
    page_id_167251659804799                  9,851

--- page_name ---
missing: 0
count : 246,745
unique: 4,546
top 5 :
    Kamala Harris                            55,503
    Donald J. Trump                          23,988
    Joe Biden                                14,822
    The Daily Scroll                         10,461
    Kamala HQ                                7,564

--- ad_id ---
missing: 0
count : 246,745
unique: 246,745
top 5 :
    0ddb025b8544e2d58e6977ad417e742a52522... 1
    86229868e6bde3661724fe02da93504bb4fb5... 1
    07b5aefc27e872e971f793e49aac38496fa62... 1
    c62978153c04116d88ead49379916855f2cb5... 1
    785e91ef18a5794565af03a6df4e7077fe1d9... 1

--- ad_creation_time